# DINOv2 + BraTS-Africa Augmented Training (Multi-Seed)

Trains a frozen DINOv2 ViT-S/14 backbone with a lightweight classification head on a binary tumour / no-tumour task, augmenting the training set with a patient-level split of BraTS-Africa slices. Run once per seed in `{12, 23, 42, 2026}` by editing `SEED` in the configuration cell.

In [ ]:
# ------------------------------------------------------------
# Configuration
# Set these paths to your local or Drive dataset locations.
# ------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')

# Root directory containing all datasets and model weights
DATA_ROOT = "PATH/TO/PROJECT"

import os

BRATS_SOURCE_DIR  = os.path.join(DATA_ROOT, "brats_africa/glioma")      # 2D BraTS-Africa T1c slices
WESTERN_TRAIN_DIR = os.path.join(DATA_ROOT, "brain_tumor_binary/Training")
WESTERN_TEST_DIR  = os.path.join(DATA_ROOT, "brain_tumor_binary/Testing")
NIGERIAN_HEALTHY  = os.path.join(DATA_ROOT, "nigerian_healthy/axial")
DINOV2_WEIGHTS    = os.path.join(DATA_ROOT, "models/dinov2_vits14_weights.pth")

EXPERIMENT_DIR    = os.path.join(DATA_ROOT, "brats_experiment")
EXP_TRAIN_DIR     = os.path.join(EXPERIMENT_DIR, "Training")
EXP_TEST_DIR      = os.path.join(EXPERIMENT_DIR, "Testing")
BRATS_HOLDOUT_DIR = os.path.join(EXPERIMENT_DIR, "brats_holdout")

# Set the seed for this run. The paper uses {12, 23, 42, 2026}.
SEED = 42
MODEL_SAVE = os.path.join(DATA_ROOT, "models", f"dinov2_brats_augmented_seed{SEED}.pth")

IMG_SIZE          = 224
BATCH_SIZE        = 32
BRATS_TRAIN_SPLIT = 0.60

In [ ]:
# Imports & Setup

import os
import random
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import cv2

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seed set to {seed}")

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- EXISTING PATHS (read-only — nothing here gets modified) ---

In [ ]:
# Build Experiment Dataset
# Western data copied once and reused.
# Only the BraTS split rebuilds when SEED changes.

def extract_patient_id(filename):
    """Extract patient ID from a BraTS slice filename.
    Example: 'BraTS-SSA-00002-000_slice90.png'  'BraTS-SSA-00002-000'
    """return filename.split('_slice')[0]

def copy_western_data_once(western_train, western_test, experiment_dir):
    """Copies western Training and Testing folders into experiment_dir.
    Skips entirely if already done — safe to call repeatedly.
    """needs_copy = False
    for split, src_root in [('Training', western_train), ('Testing', western_test)]:
        for label in ['tumor', 'notumor']:
            dst = os.path.join(experiment_dir, split, label)
            if not os.path.exists(dst) or len(os.listdir(dst)) == 0:
                needs_copy = True
                break

    if not needs_copy:
        print(" Western data already copied, skipping.")
        return

    print(" Copying western data (one-time)...")
    for split, src_root in [('Training', western_train), ('Testing', western_test)]:
        for label in ['tumor', 'notumor']:
            src = os.path.join(src_root, label)
            dst = os.path.join(experiment_dir, split, label)
            os.makedirs(dst, exist_ok=True)
            files = [f for f in os.listdir(src)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            for f in files:
                shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            print(f"{split}/{label}: {len(files)} western images copied")

def rebuild_brats_split(brats_source, experiment_dir, train_split=0.60, seed=SEED):
    """Rebuilds the BraTS split for a given seed.
    - Removes only BraTS slices from Training/tumor (prefix: 'brats_')
    - Wipes and recreates brats_holdout/
    - Leaves western data completely untouched
    """random.seed(seed)

    train_tumor_dir = os.path.join(experiment_dir, "Training", "tumor")
    holdout_dir     = os.path.join(experiment_dir, "brats_holdout")

    # --- Remove only the brats_ prefixed files from training ---
    removed = 0
    for f in os.listdir(train_tumor_dir):
        if f.startswith("brats_"):
            os.remove(os.path.join(train_tumor_dir, f))
            removed += 1
    print(f"Removed {removed} BraTS slices from previous split")

    # --- Wipe and recreate the holdout folder ---
    if os.path.exists(holdout_dir):
        shutil.rmtree(holdout_dir)
    os.makedirs(holdout_dir, exist_ok=True)

    # --- Patient-level split ---
    brats_files = sorted([
        f for f in os.listdir(brats_source)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    patient_to_slices = {}
    for f in brats_files:
        pid = extract_patient_id(f)
        patient_to_slices.setdefault(pid, []).append(f)

    unique_patients = sorted(patient_to_slices.keys())
    random.shuffle(unique_patients)

    split_idx        = int(len(unique_patients) * train_split)
    train_patients   = unique_patients[:split_idx]
    holdout_patients = unique_patients[split_idx:]

    brats_train_slices   = [s for p in train_patients   for s in patient_to_slices[p]]
    brats_holdout_slices = [s for p in holdout_patients for s in patient_to_slices[p]]

    overlap = set(train_patients) & set(holdout_patients)
    assert len(overlap) == 0, f"PATIENT LEAKAGE: {overlap}"# Copy training BraTS slices
    for f in brats_train_slices:
        shutil.copy2(
            os.path.join(brats_source, f),
            os.path.join(train_tumor_dir, f"brats_{f}")
        )

    # Copy holdout BraTS slices
    for f in brats_holdout_slices:
        shutil.copy2(
            os.path.join(brats_source, f),
            os.path.join(holdout_dir, f)
        )

    print(f"\n BraTS Patient-Level Split (seed={seed}):")
    print(f"Total patients : {len(unique_patients)}")
    print(f"Train patients : {len(train_patients)}    {len(brats_train_slices)} slices")
    print(f"Holdout patients: {len(holdout_patients)}   {len(brats_holdout_slices)} slices")
    print(f"Zero patient overlap confirmed\n")

    # Final training tumor count (western + brats)
    total_train_tumor = len(os.listdir(train_tumor_dir))
    print(f"Training/tumor total: {total_train_tumor}  "f"(western + {len(brats_train_slices)} BraTS)")

# --- Run both: western copy is skipped if already done ---
copy_western_data_once(WESTERN_TRAIN_DIR, WESTERN_TEST_DIR, EXPERIMENT_DIR)
rebuild_brats_split(BRATS_SOURCE_DIR, EXPERIMENT_DIR, train_split=BRATS_TRAIN_SPLIT, seed=SEED)

In [ ]:
# Load DINOv2 Backbone from Drive
# Pulls pretrained weights saved in the previous experiment

print(" Loading DINOv2 backbone weights from Drive...")
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=False)
backbone.load_state_dict(torch.load(DINOV2_WEIGHTS, map_location=device))

# Freeze backbone — same as before
for param in backbone.parameters():
    param.requires_grad = False

print(" Backbone loaded and frozen.")
print(f"Frozen params: {sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# Model Architecture
# Fresh head — not loading previous classifier weights
# Previous head learned western-only boundary, start clean

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

model = DINOv2Classifier(backbone).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  (head only)")
print(f"Frozen params    : {total - trainable:,}  (DINOv2 backbone)")

In [ ]:
# Data Loaders

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(contrast=0.1, brightness=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root=EXP_TRAIN_DIR,
    transform=train_transforms
)
val_dataset = datasets.ImageFolder(
    root=EXP_TEST_DIR,
    transform=val_transforms
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f"Class mapping  : {train_dataset.class_to_idx}")
print(f"Train samples  : {len(train_dataset)}")
print(f"Val samples    : {len(val_dataset)}")

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.head.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5,
    patience=3, min_lr=1e-6
)

PATIENCE       = 7
best_val_acc   = 0.0
patience_count = 0
best_epoch     = 0
history        = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print(" Starting training...\n")

for epoch in range(30):

    # --- TRAIN ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.float().to(device)

        optimizer.zero_grad()
        outputs = model(batch_X).squeeze(1)
        loss    = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * batch_X.size(0)
        preds          = (outputs > 0.5).float()
        train_correct += (preds == batch_y).sum().item()
        train_total   += batch_X.size(0)

    train_loss /= train_total
    train_acc   = train_correct / train_total

    # --- VALIDATE ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.float().to(device)

            outputs     = model(batch_X).squeeze(1)
            loss        = criterion(outputs, batch_y)
            val_loss   += loss.item() * batch_X.size(0)
            preds       = (outputs > 0.5).float()
            val_correct += (preds == batch_y).sum().item()
            val_total   += batch_X.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(
        f"Epoch {epoch+1:02d}/30 | "f"Train Acc: {train_acc:.4f} | Train Loss: {train_loss:.4f} | "f"Val Acc: {val_acc:.4f} | Val Loss: {val_loss:.4f} | "f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        best_epoch     = epoch + 1
        patience_count = 0
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'             : best_val_acc,
        }, MODEL_SAVE)
        print(f"Best model saved (val_acc: {best_val_acc:.4f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\n⏹  Early stopping at epoch {epoch+1}. Best was epoch {best_epoch}.")
            break

print(f"\n Training complete. Best val_acc: {best_val_acc:.4f} at epoch {best_epoch}")

In [ ]:
# Training Curves

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_acc'], label='Train Accuracy')
axes[0].plot(history['val_acc'],   label='Val Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history['train_loss'], label='Train Loss')
axes[1].plot(history['val_loss'],   label='Val Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.suptitle(
    'DINOv2 — BraTS-Augmented Training (60% BraTS in Training Set)',
    fontsize=12
)
plt.tight_layout()
plt.show()

In [ ]:
# Load Best Model & External Evaluation

checkpoint = torch.load(MODEL_SAVE, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Best model loaded (val_acc: {checkpoint['val_acc']:.4f})")

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def preprocess_for_dinov2(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = (img - MEAN) / STD
    img = torch.tensor(img).permute(2, 0, 1)
    img = img.unsqueeze(0).float().to(device)
    return img

def evaluate_external(model, img_dir, label, ground_truth, color):
    img_files = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    if not img_files:
        print(f"No images found in {img_dir}")
        return None, None

    preds = []
    for f in img_files:
        img = cv2.imread(os.path.join(img_dir, f))
        if img is None:
            continue
        tensor = preprocess_for_dinov2(img)
        with torch.no_grad():
            p = model(tensor).item()
        preds.append(p)

    preds   = np.array(preds)
    total   = len(preds)
    correct = np.sum(preds > 0.5) if ground_truth == 'tumor' else np.sum(preds < 0.5)
    metric  = correct / total
    metric_name = "SENSITIVITY" if ground_truth == 'tumor' else "SPECIFICITY"print(f"\n------------------------------")
    print(f"RESULTS: {label}")
    print(f"------------------------------")
    print(f"Correctly classified : {correct} / {total}")
    print(f"{metric_name:<20}: {metric:.2%}")
    print(f"Error Rate           : {1-metric:.2%}")
    print(f"------------------------------")

    plt.figure(figsize=(8, 4))
    plt.hist(preds, bins=20, range=(0, 1), color=color, edgecolor='black', alpha=0.85)
    plt.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Threshold (0.5)')
    plt.axvspan(
        0.5 if ground_truth == 'healthy' else 0.0,
        1.0 if ground_truth == 'healthy' else 0.5,
        alpha=0.07, color='red'
    )
    plt.title(f"{label}\n{metric_name}: {metric:.2%}  |  N={total}", fontsize=11)
    plt.xlabel("Tumor Probability  (0=Healthy, 1=Tumor)")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return preds, metric

# --- Run evaluations ---
print(" Evaluating on BraTS holdout (40% — unseen Nigerian tumour slices)...")
brats_preds, brats_sensitivity = evaluate_external(
    model, BRATS_HOLDOUT_DIR,
    label="BraTS-Africa Holdout — Nigerian Tumour Slices (40%)",
    ground_truth='tumor',
    color='teal'
)

print("\n Evaluating on Nigerian healthy axial scans...")
healthy_preds, nigerian_specificity = evaluate_external(
    model, NIGERIAN_HEALTHY,
    label="Nigerian Healthy Scans — Axial",
    ground_truth='healthy',
    color='steelblue'
)

# --- Summary ---
print("\n" + "="*50)
print("EXPERIMENT SUMMARY — DINOv2 + BraTS-Augmented Training")
print("="*50)
print(f"{'BraTS Holdout Sensitivity':<30}: {brats_sensitivity:.2%}  (n=~60)")
print(f"{'Nigerian Healthy Specificity':<30}: {nigerian_specificity:.2%}  (n=25)")
print(f"{'Nigerian Healthy FPR':<30}: {1-nigerian_specificity:.2%}")
print("="*50)
print("\n  Note: Both evaluation sets are small. Treat results as directional.")

# --- Comparison with previous experiment ---
print("\n COMPARISON TABLE")
print("-"*55)
print(f"{'Model':<35} {'Sensitivity':>12} {'Specificity':>12}")
print("-"*55)
print(f"{'MobileNetV2 (skull-stripped, original)':<35} {'85.67%':>12} {'Unknown':>12}")
print(f"{'DINOv2 (western only)':<35} {'14.67%':>12} {'92.00%':>12}")
print(f"{'DINOv2 (BraTS-augmented)':<35} {f'{brats_sensitivity:.2%}':>12} {f'{nigerian_specificity:.2%}':>12}")
print("-"*55)